In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (

    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.layers import LSTM,Bidirectional
from tensorflow.keras.optimizers import (
    Adam,
    RMSprop
)

In [2]:
data=pd.read_csv('/kaggle/input/datasets/manishavelamani/dataset/PJME_preprocessd.csv',parse_dates=['Datetime'],index_col='Datetime')

In [3]:
data.columns

Index(['PJME_MW', 'PJME_MW_Scaled', 'Hour', 'Day', 'Week', 'Month',
       'DayOfWeek', 'Weekend', 'Lag_1', 'Lag_24', 'Lag_48', 'Lag_168',
       'RollingMean_24', 'RollingStd_24', 'RollingMean_168'],
      dtype='object')

In [4]:
from sklearn.preprocessing import MinMaxScaler

features = [
    'PJME_MW_Scaled',
    'Hour',
    'Day',
    'Week',
    'Month',
    'DayOfWeek',
    'Weekend',
    'Lag_1',
    'Lag_24',
    'Lag_48',
    'Lag_168',
    'RollingMean_24',
    'RollingStd_24',
    'RollingMean_168'
]

target = 'PJME_MW_Scaled'

# Select input features
multivariate_data = data[features].copy()

# Scale ALL input features
feature_scaler = MinMaxScaler()

multivariate_data_scaled = pd.DataFrame(
    feature_scaler.fit_transform(multivariate_data),
    columns=features,
    index=multivariate_data.index
)

multivariate_data_scaled.head()

,PJME_MW_Scaled,Hour,Day,Week,Month,DayOfWeek,Weekend,Lag_1,Lag_24,Lag_48,Lag_168,RollingMean_24,RollingStd_24,RollingMean_168
Datetime,,,,,,,,,,,,,,
2002-01-08 01:00:00,0.433011,0.043478,0.233333,0.019231,0.0,0.166667,0.0,0.486937,0.353052,0.360420,0.462358,0.535733,0.407201,0.421362
2002-01-08 02:00:00,0.409021,0.086957,0.233333,0.019231,0.0,0.166667,0.0,0.433011,0.325625,0.329371,0.427439,0.539989,0.386780,0.421179
2002-01-08 03:00:00,0.399889,0.130435,0.233333,0.019231,0.0,0.166667,0.0,0.409021,0.315255,0.319960,0.399331,0.544309,0.363683,0.421185
2002-01-08 04:00:00,0.405058,0.173913,0.233333,0.019231,0.0,0.166667,0.0,0.399889,0.316029,0.315750,0.385154,0.548853,0.338064,0.421382
2002-01-08 05:00:00,0.427316,0.217391,0.233333,0.019231,0.0,0.166667,0.0,0.405058,0.336522,0.319496,0.390045,0.553487,0.312793,0.421752


In [5]:
train_size = int(len(multivariate_data_scaled) * 0.70)
val_size = int(len(multivariate_data_scaled) * 0.10)

train = multivariate_data_scaled.iloc[:train_size]
validation = multivariate_data_scaled.iloc[train_size:train_size + val_size]
test = multivariate_data_scaled.iloc[train_size + val_size:]

In [6]:
def create_multivariate_sequences(df, sequence_length, forecast_horizon=24, target_col='PJME_MW_Scaled'):
    X = []
    y = []

    values = df.values
    target_index = df.columns.get_loc(target_col)

    for i in range(len(df) - sequence_length - forecast_horizon + 1):
        X.append(values[i:i + sequence_length])

        y.append(
            values[
                i + sequence_length:
                i + sequence_length + forecast_horizon,
                target_index
            ]
        )

    return np.array(X), np.array(y)

In [7]:
sequence_length = 168

X_train, y_train = create_multivariate_sequences(train, sequence_length)
X_val, y_val = create_multivariate_sequences(validation, sequence_length)
X_test, y_test = create_multivariate_sequences(test, sequence_length)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)

print('X_val:', X_val.shape)
print('y_val:', y_val.shape)

print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train: (101465, 168, 14)
y_train: (101465, 24)
X_val: (14331, 168, 14)
y_val: (14331, 24)
X_test: (28855, 168, 14)
y_test: (28855, 24)


In [8]:
from sklearn.preprocessing import MinMaxScaler
# Create scaler using the original MW values
scaler = MinMaxScaler()
scaler.fit(data[['PJME_MW']])

MinMaxScaler()

In [9]:
def evaluate_model(model, X_test, y_test, scaler, sequence_length, model_name):
    predictions = model.predict(X_test, verbose=0)
    # Convert scaled values back to original MW values
    pred_original = scaler.inverse_transform(predictions.reshape(-1, 1)).reshape(predictions.shape)
    y_original = scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)
    mae = mean_absolute_error(y_original.flatten(),pred_original.flatten())
    mse = mean_squared_error(y_original.flatten(),pred_original.flatten())
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_original.flatten(),pred_original.flatten()) * 100
    r2 = r2_score(y_original.flatten(),pred_original.flatten())
    bias = np.mean(pred_original.flatten() - y_original.flatten())
    return {
        'Sequence Length': sequence_length,
        'Model': model_name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2,
        'Bias': bias
    }

In [10]:
bilstm_phase7 = Sequential([
    Bidirectional(
        LSTM(32),
        input_shape=(sequence_length, X_train.shape[2])
    ),
    Dense(24)
])

bilstm_phase7.compile(
    optimizer='adam',
    loss='mse'
)

bilstm_phase7.summary()

bilstm_baseline_history = bilstm_phase7.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

baseline_bilstm_result = evaluate_model(
    bilstm_phase7,
    X_test,
    y_test,
    scaler,
    168,
    'Baseline Bidirectional LSTM (Multivariate)'
)

I0000 00:00:1786481682.710356      59 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 64)             │        12,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,592 (53.09 KB)

 Trainable params: 13,592 (53.09 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 15ms/step - loss: 0.0070 - val_loss: 0.0040
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0032 - val_loss: 0.0036
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0029 - val_loss: 0.0033
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0027 - val_loss: 0.0032
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0026 - val_loss: 0.0031
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 46s 15ms/step - loss: 0.0026 - val_loss: 0.0031
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 46s 15ms/step - loss: 0.0025 - val_loss: 0.0030
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 46s 15ms/step - loss: 0.0024 - val_loss: 0.0030
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 46s 15ms/step - loss: 0.0024 - val_loss: 0.0030
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0023 - val_loss: 0.0029


In [11]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)
bilstm_early = Sequential([
    Bidirectional(
        LSTM(32),
        input_shape=(sequence_length, X_train.shape[2])
    ),
    Dense(24)
])

bilstm_early.compile(
    optimizer='adam',
    loss='mse'
)

bilstm_early_history = bilstm_early.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)

bilstm_early_result = evaluate_model(
    bilstm_early,
    X_test,
    y_test,
    scaler,
    168,
    'Bidirectional LSTM + EarlyStopping'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 15ms/step - loss: 0.0070 - val_loss: 0.0041
Epoch 2/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 48s 15ms/step - loss: 0.0032 - val_loss: 0.0037
Epoch 3/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 46s 15ms/step - loss: 0.0030 - val_loss: 0.0035
Epoch 4/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 46s 15ms/step - loss: 0.0028 - val_loss: 0.0033
Epoch 5/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 46s 15ms/step - loss: 0.0027 - val_loss: 0.0033
Epoch 6/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 46s 15ms/step - loss: 0.0026 - val_loss: 0.0032
Epoch 7/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 46s 15ms/step - loss: 0.0025 - val_loss: 0.0031
Epoch 8/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0024 - val_loss: 0.0031
Epoch 9/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 46s 15ms/step - loss: 0.0024 - val_loss: 0.0030
Epoch 10/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0023 - val_loss: 0.0031
Epoch 11/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0023 - val_loss: 0.0032
Epoch 12

In [12]:
bilstm_dropout = Sequential([
    Bidirectional(
        LSTM(32),
        input_shape=(sequence_length, X_train.shape[2])
    ),
    Dropout(0.2),
    Dense(24)
])

bilstm_dropout.compile(
    optimizer='adam',
    loss='mse'
)

bilstm_dropout_history = bilstm_dropout.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

bilstm_dropout_result = evaluate_model(
    bilstm_dropout,
    X_test,
    y_test,
    scaler,
    168,
    'Bidirectional LSTM + Dropout'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 15ms/step - loss: 0.0114 - val_loss: 0.0042
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0043 - val_loss: 0.0037
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0038 - val_loss: 0.0035
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0036 - val_loss: 0.0034
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0035 - val_loss: 0.0035
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0034 - val_loss: 0.0032
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0034 - val_loss: 0.0033
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0033 - val_loss: 0.0031
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0033 - val_loss: 0.0032
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - loss: 0.0032 - val_loss: 0.0032


In [13]:
bilstm_more_units = Sequential([
    Bidirectional(
        LSTM(64),
        input_shape=(sequence_length, X_train.shape[2])
    ),
    Dense(24)
])

bilstm_more_units.compile(
    optimizer='adam',
    loss='mse'
)

bilstm_more_units_history = bilstm_more_units.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

bilstm_more_units_result = evaluate_model(
    bilstm_more_units,
    X_test,
    y_test,
    scaler,
    168,
    'Bidirectional LSTM + More Units'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 54s 17ms/step - loss: 0.0055 - val_loss: 0.0040
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 17ms/step - loss: 0.0031 - val_loss: 0.0037
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 53s 17ms/step - loss: 0.0028 - val_loss: 0.0032
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 53s 17ms/step - loss: 0.0026 - val_loss: 0.0032
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 17ms/step - loss: 0.0024 - val_loss: 0.0031
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 17ms/step - loss: 0.0023 - val_loss: 0.0030
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 17ms/step - loss: 0.0022 - val_loss: 0.0030
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 17ms/step - loss: 0.0022 - val_loss: 0.0028
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 16ms/step - loss: 0.0021 - val_loss: 0.0032
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 17ms/step - loss: 0.0020 - val_loss: 0.0029


In [14]:
bilstm_batchnorm = Sequential([
    Bidirectional(
        LSTM(64),
        input_shape=(sequence_length, X_train.shape[2])
    ),
    BatchNormalization(),
    Dense(24)
])

bilstm_batchnorm.compile(
    optimizer='adam',
    loss='mse'
)

bilstm_batchnorm_history = bilstm_batchnorm.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

bilstm_batchnorm_result = evaluate_model(
    bilstm_batchnorm,
    X_test,
    y_test,
    scaler,
    168,
    'Bidirectional LSTM + BatchNormalization'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 57s 17ms/step - loss: 0.0150 - val_loss: 0.0087
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 54s 17ms/step - loss: 0.0048 - val_loss: 0.0067
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 54s 17ms/step - loss: 0.0045 - val_loss: 0.0066
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 54s 17ms/step - loss: 0.0044 - val_loss: 0.0053
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 55s 17ms/step - loss: 0.0043 - val_loss: 0.0041
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 54s 17ms/step - loss: 0.0042 - val_loss: 0.0044
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 54s 17ms/step - loss: 0.0041 - val_loss: 0.0037
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 54s 17ms/step - loss: 0.0040 - val_loss: 0.0038
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 54s 17ms/step - loss: 0.0040 - val_loss: 0.0039
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 54s 17ms/step - loss: 0.0039 - val_loss: 0.0040


In [15]:
bilstm_rmsprop = Sequential([
    Bidirectional(
        LSTM(64),
        input_shape=(sequence_length, X_train.shape[2])
    ),
    Dense(24)
])

bilstm_rmsprop.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss='mse'
)

bilstm_rmsprop_history = bilstm_rmsprop.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop]
)

bilstm_rmsprop_result = evaluate_model(
    bilstm_rmsprop,
    X_test,
    y_test,
    scaler,
    168,
    'Bidirectional LSTM + RMSprop'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 53s 16ms/step - loss: 0.0057 - val_loss: 0.0041
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 51s 16ms/step - loss: 0.0033 - val_loss: 0.0040
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 51s 16ms/step - loss: 0.0031 - val_loss: 0.0036
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 51s 16ms/step - loss: 0.0030 - val_loss: 0.0036
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 51s 16ms/step - loss: 0.0029 - val_loss: 0.0035


In [16]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

bilstm_lr = Sequential([
    Bidirectional(
        LSTM(64),
        input_shape=(sequence_length, X_train.shape[2])
    ),
    Dense(24)
])

bilstm_lr.compile(
    optimizer='adam',
    loss='mse'
)

bilstm_lr_history = bilstm_lr.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[lr_scheduler]
)

bilstm_lr_result = evaluate_model(
    bilstm_lr,
    X_test,
    y_test,
    scaler,
    168,
    'Bidirectional LSTM + LR Scheduler'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 55s 17ms/step - loss: 0.0052 - val_loss: 0.0038 - learning_rate: 0.0010
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 17ms/step - loss: 0.0030 - val_loss: 0.0034 - learning_rate: 0.0010
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 17ms/step - loss: 0.0027 - val_loss: 0.0035 - learning_rate: 0.0010
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 17ms/step - loss: 0.0026 - val_loss: 0.0031 - learning_rate: 0.0010
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 17ms/step - loss: 0.0025 - val_loss: 0.0030 - learning_rate: 0.0010
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 17ms/step - loss: 0.0023 - val_loss: 0.0031 - learning_rate: 0.0010
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 16ms/step - loss: 0.0023 - val_loss: 0.0030 - learning_rate: 0.0010
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 17ms/step - loss: 0.0022 - val_loss: 0.0030 - learning_rate: 0.0010
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 16ms/step - loss: 0.0021 - val_loss: 0.002

In [17]:
bilstm_bs16 = Sequential([
    Bidirectional(
        LSTM(64),
        input_shape=(sequence_length, X_train.shape[2])
    ),
    Dense(24)
])

bilstm_bs16.compile(
    optimizer='adam',
    loss='mse'
)

bilstm_bs16_history = bilstm_bs16.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=16
)

bilstm_bs16_result = evaluate_model(
    bilstm_bs16,
    X_test,
    y_test,
    scaler,
    168,
    'Bidirectional LSTM + Batch Size 16'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 79s 12ms/step - loss: 0.0043 - val_loss: 0.0036
Epoch 2/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 77s 12ms/step - loss: 0.0029 - val_loss: 0.0032
Epoch 3/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 77s 12ms/step - loss: 0.0026 - val_loss: 0.0032
Epoch 4/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 77s 12ms/step - loss: 0.0024 - val_loss: 0.0030
Epoch 5/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 77s 12ms/step - loss: 0.0023 - val_loss: 0.0031
Epoch 6/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 77s 12ms/step - loss: 0.0022 - val_loss: 0.0028
Epoch 7/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 77s 12ms/step - loss: 0.0021 - val_loss: 0.0029
Epoch 8/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 77s 12ms/step - loss: 0.0021 - val_loss: 0.0030
Epoch 9/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 77s 12ms/step - loss: 0.0020 - val_loss: 0.0030
Epoch 10/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 77s 12ms/step - loss: 0.0019 - val_loss: 0.0029


In [18]:
bilstm_bs64 = Sequential([
    Bidirectional(
        LSTM(64),
        input_shape=(sequence_length, X_train.shape[2])
    ),
    Dense(24)
])

bilstm_bs64.compile(
    optimizer='adam',
    loss='mse'
)

bilstm_bs64_history = bilstm_bs64.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

bilstm_bs64_result = evaluate_model(
    bilstm_bs64,
    X_test,
    y_test,
    scaler,
    168,
    'Bidirectional LSTM + Batch Size 64'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 30s 18ms/step - loss: 0.0068 - val_loss: 0.0044
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0032 - val_loss: 0.0037
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0030 - val_loss: 0.0036
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0028 - val_loss: 0.0033
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0027 - val_loss: 0.0034
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0026 - val_loss: 0.0035
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0025 - val_loss: 0.0031
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0024 - val_loss: 0.0030
Epoch 9/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0023 - val_loss: 0.0029
Epoch 10/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0022 - val_loss: 0.0029


In [19]:
bilstm_two_layers = Sequential([
    Bidirectional(
        LSTM(
            64,
            return_sequences=True
        ),
        input_shape=(sequence_length, X_train.shape[2])
    ),
    Bidirectional(
        LSTM(32)
    ),
    Dense(24)
])

bilstm_two_layers.compile(
    optimizer='adam',
    loss='mse'
)

bilstm_two_layers.summary()

bilstm_two_layers_history = bilstm_two_layers.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

bilstm_two_layers_result = evaluate_model(
    bilstm_two_layers,
    X_test,
    y_test,
    scaler,
    168,
    'Bidirectional LSTM + Two Layers'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_9 (Bidirectional) │ (None, 168, 128)       │        40,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_10                │ (None, 64)             │        41,216 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83,224 (325.09 KB)

 Trainable params: 83,224 (325.09 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 104s 32ms/step - loss: 0.0051 - val_loss: 0.0037
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 101s 32ms/step - loss: 0.0029 - val_loss: 0.0035
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 100s 32ms/step - loss: 0.0027 - val_loss: 0.0035
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 101s 32ms/step - loss: 0.0025 - val_loss: 0.0032
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 101s 32ms/step - loss: 0.0023 - val_loss: 0.0031
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 101s 32ms/step - loss: 0.0022 - val_loss: 0.0031
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 101s 32ms/step - loss: 0.0020 - val_loss: 0.0032
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 101s 32ms/step - loss: 0.0019 - val_loss: 0.0033
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 101s 32ms/step - loss: 0.0017 - val_loss: 0.0034
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 101s 32ms/step - loss: 0.0016 - val_loss: 0.0036


In [20]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam

# ---------------------------------------------------
# Build Bidirectional LSTM model for hyperparameter tuning
# ---------------------------------------------------
def build_bilstm(hp):

    model = Sequential()

    model.add(
        Bidirectional(
            LSTM(
                units=hp.Choice(
                    'bilstm_units',
                    values=[32, 64, 128]
                )
            ),
            input_shape=(sequence_length, X_train.shape[2])
        )
    )

    model.add(
        Dropout(
            hp.Choice(
                'dropout_rate',
                values=[0.0, 0.2, 0.3]
            )
        )
    )

    model.add(
        Dense(
            units=hp.Choice(
                'dense_units',
                values=[32, 64, 128]
            ),
            activation='relu'
        )
    )

    model.add(Dense(24))

    learning_rate = hp.Choice(
        'learning_rate',
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss='mse',
        metrics=['mae']
    )

    return model


# ---------------------------------------------------
# Hyperparameter search
# ---------------------------------------------------
tuner = kt.RandomSearch(
    build_bilstm,
    objective='val_loss',
    max_trials=3,
    executions_per_trial=1,
    directory='hyperparameter_tuning',
    project_name='bilstm_168_to_24'
)

tuner.search(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)


# ---------------------------------------------------
# Get best hyperparameters
# ---------------------------------------------------
best_hp = tuner.get_best_hyperparameters(1)[0]

print('Best Bi-LSTM Units:', best_hp.get('bilstm_units'))
print('Best Dropout:', best_hp.get('dropout_rate'))
print('Best Dense Units:', best_hp.get('dense_units'))
print('Best Learning Rate:', best_hp.get('learning_rate'))


# ---------------------------------------------------
# Rebuild and retrain the best model
# ---------------------------------------------------
final_bilstm = build_bilstm(best_hp)

final_bilstm_history = final_bilstm.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)


# ---------------------------------------------------
# Evaluate the retrained final model
# ---------------------------------------------------
final_bilstm_result = evaluate_model(
    final_bilstm,
    X_test,
    y_test,
    scaler,
    168,
    'Final Tuned Bi-LSTM'
)

pd.DataFrame([final_bilstm_result])


# ---------------------------------------------------
# Save the final tuned model
# ---------------------------------------------------
final_bilstm.save('/kaggle/working/bilstm168_phase6_final_tuned.keras')

Trial 3 Complete [00h 09m 10s]
val_loss: 0.0029559244867414236

Best val_loss So Far: 0.0028836133424192667
Total elapsed time: 00h 29m 06s
Best Bi-LSTM Units: 128
Best Dropout: 0.2
Best Dense Units: 32
Best Learning Rate: 0.001
Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 65s 20ms/step - loss: 0.0076 - mae: 0.0594 - val_loss: 0.0040 - val_mae: 0.0461
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 63s 20ms/step - loss: 0.0035 - mae: 0.0441 - val_loss: 0.0035 - val_mae: 0.0433
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 63s 20ms/step - loss: 0.0031 - mae: 0.0414 - val_loss: 0.0035 - val_mae: 0.0446
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 63s 20ms/step - loss: 0.0029 - mae: 0.0398 - val_loss: 0.0032 - val_mae: 0.0410
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 63s 20ms/step - loss: 0.0028 - mae: 0.0386 - val_loss: 0.0031 - val_mae: 0.0401
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 63s 20ms/step - loss: 0.0027 - mae: 0.0377 - val_loss: 0.0031 - val_mae: 0.0397
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━

In [21]:
comparison_bilstm = pd.DataFrame([
    baseline_bilstm_result,
    bilstm_early_result,
    bilstm_dropout_result,
    bilstm_more_units_result,
    bilstm_batchnorm_result,
    bilstm_rmsprop_result,
    bilstm_lr_result,
    bilstm_bs16_result,
    bilstm_bs64_result,
    bilstm_two_layers_result,
    final_bilstm_result
])

comparison_bilstm = comparison_bilstm.sort_values('RMSE').reset_index(drop=True)

comparison_bilstm

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,Baseline Bidirectional LSTM (Multivariate),1276.378402,3.259922e+06,1805.525407,4.045682,0.918346,55.596732
1,168,Bidirectional LSTM + EarlyStopping,1282.892348,3.323285e+06,1822.987868,4.078960,0.916759,106.958338
2,168,Bidirectional LSTM + Batch Size 64,1289.043797,3.333111e+06,1825.680893,4.126460,0.916513,267.272873
3,168,Bidirectional LSTM + More Units,1280.037649,3.345686e+06,1829.121660,4.083994,0.916198,211.411292
4,168,Final Tuned Bi-LSTM,1307.866460,3.414100e+06,1847.728232,4.165002,0.914484,130.555495
5,168,Bidirectional LSTM + Dropout,1364.321005,3.517484e+06,1875.495667,4.412656,0.911895,424.459715
6,168,Bidirectional LSTM + LR Scheduler,1331.509697,3.528458e+06,1878.418921,4.246586,0.911620,211.789270
7,168,Bidirectional LSTM + Batch Size 16,1350.696415,3.552622e+06,1884.840067,4.318001,0.911015,206.458140
8,168,Bidirectional LSTM + Two Layers,1409.238557,4.009280e+06,2002.318686,4.481538,0.899577,55.266596
9,168,Bidirectional LSTM + BatchNormalization,1532.464895,4.370785e+06,2090.642340,4.963761,0.890522,304.374338


In [22]:
comparison_bilstm.to_csv('/kaggle/working/Comparisonlstm_bilstm_csv')
print("saved")

saved
